# Delhi NCR Climate Analysis

**Submitted by:** Aditya Kumar  
**Class:** AIML 01  
**Institute:** ABES Engineering College

A beginner-level data analysis project using **Pandas, NumPy and Matplotlib**.

Keep `delhi_ncr_climate_raw_data.csv` in the same folder as this notebook before running it.

## 1. Install and import libraries

If these libraries are not installed, run the installation command in a terminal:  
`pip install pandas numpy matplotlib`

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT_FILE = "delhi_ncr_climate_raw_data.csv"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Libraries imported successfully!')

## 2. Load and explore the CSV dataset

In [ ]:
data = pd.read_csv(INPUT_FILE)

print('First five rows:')
display(data.head())
print('Dataset shape (rows, columns):', data.shape)
print('\nColumn names:')
print(data.columns.tolist())

## 3. Clean the data

Convert dates and numeric columns, handle missing-value markers, and standardise city names.

In [ ]:
def clean_numeric(series):
    values = series.astype(str).str.strip()
    values = values.str.replace(r"(?i)\\s*mm$", "", regex=True)
    values = values.str.replace(",", "", regex=False)
    values = values.replace({
        "": np.nan, "nan": np.nan, "None": np.nan, "missing": np.nan,
        "--": np.nan, "NA": np.nan, "N/A": np.nan, "null": np.nan
    })
    numbers = pd.to_numeric(values, errors="coerce")
    return numbers.mask(numbers == -999)

def standardize_city(value):
    if pd.isna(value):
        return np.nan
    name = str(value).strip().lower()
    city_names = {
        "delhi": "Delhi", "new delhi": "Delhi", "n. delhi": "Delhi",
        "ghaziabad": "Ghaziabad", "gzb": "Ghaziabad",
        "faridabad": "Faridabad", "gurgaon": "Gurugram",
        "gurugram": "Gurugram", "noida": "Noida",
        "greater noida": "Greater Noida"
    }
    return city_names.get(name, str(value).strip().title())

date_text = data["Date"].astype(str).str.strip()
data["Date"] = pd.to_datetime(date_text, format="mixed", dayfirst=False, errors="coerce")
failed_dates = data["Date"].isna()
if failed_dates.any():
    data.loc[failed_dates, "Date"] = pd.to_datetime(
        date_text[failed_dates], format="mixed", dayfirst=True, errors="coerce"
    )

numeric_columns = [
    "Max Temp (C)", "Min Temp(°C)", "Humidity %", "Rainfall_mm",
    "Wind Speed (km/h)", "PM2.5 (ug/m3)", "PM10", "AQI", "Visibility (km)"
]
for column in numeric_columns:
    data[column] = clean_numeric(data[column])

data["City"] = data["City"].apply(standardize_city)
data["AQI_Category"] = data["AQI_Category"].astype("string").str.strip().str.lower()
category_names = {
    "good": "Good", "satisfactory": "Satisfactory", "moderate": "Moderate",
    "poor": "Poor", "very poor": "Very Poor", "verypoor": "Very Poor",
    "severe": "Severe"
}
data["AQI_Category"] = data["AQI_Category"].map(category_names)

# Treat values outside broad plausible ranges as missing.
valid_ranges = {
    "Max Temp (C)": (-10, 60), "Min Temp(°C)": (-15, 50),
    "Humidity %": (0, 100), "Rainfall_mm": (0, 1000),
    "Wind Speed (km/h)": (0, 200), "PM2.5 (ug/m3)": (0, 2000),
    "PM10": (0, 3000), "AQI": (0, 1000), "Visibility (km)": (0, 100)
}
for column, limits in valid_ranges.items():
    data.loc[~data[column].between(*limits), column] = np.nan

data["Month"] = data["Date"].dt.month_name()
print('Data cleaning completed.')
display(data.head())

## 4. Check missing values and calculate basic statistics

In [ ]:
print('Missing values in each column:')
display(data.isna().sum().to_frame('Missing values'))

print('Average maximum temperature:', round(data['Max Temp (C)'].mean(), 2), '°C')
print('Average minimum temperature:', round(data['Min Temp(°C)'].mean(), 2), '°C')
print('Average AQI:', round(data['AQI'].mean(), 2))
print('Average PM2.5:', round(data['PM2.5 (ug/m3)'].mean(), 2), 'µg/m³')

aqi_values = data['AQI'].dropna().to_numpy()
if len(aqi_values) > 0:
    print('Median AQI using NumPy:', round(np.median(aqi_values), 2))

## 5. AQI category and city-wise analysis

In [ ]:
print('Number of records in each AQI category:')
display(data['AQI_Category'].value_counts().to_frame('Records'))

print('Average AQI by city:')
city_aqi = data.groupby('City')['AQI'].mean().sort_values(ascending=False)
display(city_aqi.to_frame('Average AQI'))

## 6. Monthly climate analysis

In [ ]:
monthly = data.groupby('Month')[['Max Temp (C)', 'AQI', 'Rainfall_mm']].mean()
month_order = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]
monthly = monthly.reindex(month_order)
display(monthly)

## 7. Visualisation using Matplotlib

In [ ]:
# Graph 1: AQI category distribution
data['AQI_Category'].value_counts().plot(kind='bar', figsize=(8, 5), title='AQI Category Distribution')
plt.xlabel('AQI Category')
plt.ylabel('Number of Records')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'aqi_category_distribution.png'))
plt.show()
plt.close()

In [ ]:
# Graph 2: Average maximum temperature by month
monthly['Max Temp (C)'].plot(kind='line', marker='o', figsize=(8, 5), title='Average Maximum Temperature by Month')
plt.xlabel('Month')
plt.ylabel('Temperature (°C)')
plt.xticks(rotation=35, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'monthly_average_temperature.png'))
plt.show()
plt.close()

In [ ]:
# Graph 3: Average AQI by city
city_aqi.sort_values().plot(kind='barh', figsize=(8, 5), title='Average AQI by City')
plt.xlabel('Average AQI')
plt.ylabel('City')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'average_aqi_by_city.png'))
plt.show()
plt.close()

In [ ]:
# Graph 4: PM2.5 compared with AQI
plt.figure(figsize=(8, 5))
plt.scatter(data['PM2.5 (ug/m3)'], data['AQI'], alpha=0.6)
plt.title('PM2.5 vs AQI')
plt.xlabel('PM2.5 (µg/m³)')
plt.ylabel('AQI')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'pm25_vs_aqi.png'))
plt.show()
plt.close()

In [ ]:
# Graph 5: Average rainfall by month
monthly['Rainfall_mm'].plot(kind='bar', figsize=(8, 5), title='Average Rainfall by Month')
plt.xlabel('Month')
plt.ylabel('Rainfall (mm)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'monthly_average_rainfall.png'))
plt.show()
plt.close()

## 8. Save the cleaned dataset

In [ ]:
cleaned_file = os.path.join(OUTPUT_DIR, 'delhi_ncr_climate_cleaned.csv')
data.to_csv(cleaned_file, index=False)
print('Project completed!')
print('Cleaned CSV and graphs are saved in:', OUTPUT_DIR)

## 9. Conclusion

This project used Pandas to load, clean and summarise climate data, NumPy to calculate a numerical statistic, and Matplotlib to create visualisations. The results describe only the records in the supplied dataset; they should not be treated as an official weather or air-quality forecast.